In [3]:
import json
import pandas as pd

INPUT_FILE = "coverage_results_rich.json"

OUTPUT_CSV_MAIN = "coverage_model_comparison.csv"
OUTPUT_CSV_THRESHOLDS = "coverage_threshold_robustness.csv"

THRESHOLDS = [0.6, 0.7, 0.8]


# ---------------------------
# LOAD RESULTS
# ---------------------------
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


# ---------------------------
# MODEL NAME NORMALIZATION
# ---------------------------
def short_name(model_name):
    return model_name.split("/")[-1]


# ---------------------------
# TABLE 1: MAIN COMPARISON (T=0.7)
# ---------------------------
main_rows = []

for model_name, results in data.items():
    df = pd.DataFrame(results)

    title_col = "title_coverage_0.7"
    description_col = "description_coverage_0.7"

    main_rows.append({
        "Embedding Model": short_name(model_name),

        "Title Semantic Completeness Mean": round(df[title_col].mean(), 4),
        "Title Semantic Completeness Median": round(df[title_col].median(), 4),

        "Description Semantic Completeness Mean": round(df[description_col].mean(), 4),
        "Description Semantic Completeness Median": round(df[description_col].median(), 4)
    })

main_df = pd.DataFrame(main_rows)
main_df = main_df.sort_values("Embedding Model")

print("\nSemantic Completeness Robustness Comparison (T=0.7)\n")
print(main_df.to_string(index=False))

main_df.to_csv(OUTPUT_CSV_MAIN, index=False)


# ---------------------------
# TABLE 2: THRESHOLD ROBUSTNESS
# ---------------------------
threshold_rows = []

# easier access
model_frames = {
    short_name(model_name): pd.DataFrame(results)
    for model_name, results in data.items()
}

for t in THRESHOLDS:
    threshold_rows.append({
        "Threshold": t,

        "MiniLM Title": round(
            model_frames["all-MiniLM-L6-v2"][f"title_coverage_{t}"].mean(),
            4
        ),

        "MiniLM Description": round(
            model_frames["all-MiniLM-L6-v2"][f"description_coverage_{t}"].mean(),
            4
        ),

        "MPNet Title": round(
            model_frames["all-mpnet-base-v2"][f"title_coverage_{t}"].mean(),
            4
        ),

        "MPNet Description": round(
            model_frames["all-mpnet-base-v2"][f"description_coverage_{t}"].mean(),
            4
        )
    })

threshold_df = pd.DataFrame(threshold_rows)

print("\nSemantic Completeness Threshold Robustness Across Embedding Models\n")
print(threshold_df.to_string(index=False))

threshold_df.to_csv(OUTPUT_CSV_THRESHOLDS, index=False)


# ---------------------------
# DONE
# ---------------------------
print("\nCSV files saved:")
print(f"- {OUTPUT_CSV_MAIN}")
print(f"- {OUTPUT_CSV_THRESHOLDS}")


Semantic Completeness Robustness Comparison (T=0.7)

  Embedding Model  Title Semantic Completeness Mean  Title Semantic Completeness Median  Description Semantic Completeness Mean  Description Semantic Completeness Median
 all-MiniLM-L6-v2                            0.2912                                 0.2                                  0.2089                                       0.2
all-mpnet-base-v2                            0.2542                                 0.2                                  0.1681                                       0.2

Semantic Completeness Threshold Robustness Across Embedding Models

 Threshold  MiniLM Title  MiniLM Description  MPNet Title  MPNet Description
       0.6        0.3735              0.2705       0.3451             0.2532
       0.7        0.2912              0.2089       0.2542             0.1681
       0.8        0.1959              0.1341       0.1682             0.1117

CSV files saved:
- coverage_model_comparison.csv
- coverag